In [1]:
!unzip /content/datasound.zip

Archive:  /content/datasound.zip
   creating: datasound/
   creating: datasound/glass_breaking/
  inflating: datasound/glass_breaking/glass_clip_0.wav  
  inflating: datasound/glass_breaking/glass_clip_1.wav  
  inflating: datasound/glass_breaking/glass_clip_10.wav  
  inflating: datasound/glass_breaking/glass_clip_100.wav  
  inflating: datasound/glass_breaking/glass_clip_101.wav  
  inflating: datasound/glass_breaking/glass_clip_102.wav  
  inflating: datasound/glass_breaking/glass_clip_103.wav  
  inflating: datasound/glass_breaking/glass_clip_104.wav  
  inflating: datasound/glass_breaking/glass_clip_105.wav  
  inflating: datasound/glass_breaking/glass_clip_106.wav  
  inflating: datasound/glass_breaking/glass_clip_107.wav  
  inflating: datasound/glass_breaking/glass_clip_108.wav  
  inflating: datasound/glass_breaking/glass_clip_109.wav  
  inflating: datasound/glass_breaking/glass_clip_11.wav  
  inflating: datasound/glass_breaking/glass_clip_110.wav  
  inflating: datasound/gl

In [32]:
import os
import json
import numpy as np
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split

In [33]:
DATA_DIR = "C:/Users/Jahnavi P Utham/OneDrive/Desktop/datasets"

In [34]:
CLASS_NAMES = sorted(os.listdir(DATA_DIR))
CLASS_NAMES = [c for c in CLASS_NAMES if os.path.isdir(os.path.join(DATA_DIR, c))]

print("Classes found:", CLASS_NAMES)

paths = []
labels = []

for idx, class_name in enumerate(CLASS_NAMES):
    class_folder = os.path.join(DATA_DIR, class_name)
    for file in os.listdir(class_folder):
        if file.endswith(".wav"):
            paths.append(os.path.join(class_folder, file))
            labels.append(idx)

paths = np.array(paths)
labels = np.array(labels)

print("Total audio files:", len(paths))

Classes found: ['accident_clips', 'glass_clips', 'gunshot', 'non_emergency', 'scream', 'siren']
Total audio files: 3967


In [35]:
SR = 22050
DURATION = 4
N_MELS = 128

def audio_to_img(path):
    y, _ = librosa.load(path, sr=SR, mono=True)

    
    target_len = SR * DURATION
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]

    
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=N_MELS, n_fft=2048, hop_length=512)
    S_db = librosa.power_to_db(S, ref=np.max)

    
    S_db = (S_db - S_db.min()) / (S_db.max() - S_db.min() + 1e-9)

    
    img = S_db.astype(np.float32)[..., np.newaxis]          
    img = tf.image.resize(img, (224, 224)).numpy()         
    img = np.repeat(img, 3, axis=-1)                        

    
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img * 255.0)

    return img.astype(np.float32)

In [36]:
X_train, X_val, y_train, y_val = train_test_split(
    paths, labels, test_size=0.2, random_state=42, stratify=labels
)

y_train = y_train.astype(np.int32)  
y_val   = y_val.astype(np.int32)

print("Train:", len(X_train), "Val:", len(X_val))
import numpy as np

np.save("X_val.npy", X_val)
np.save("y_val.npy", y_val)


Train: 3173 Val: 794


In [37]:
NUM_CLASSES = len(CLASS_NAMES)
BATCH_SIZE = 32

def py_load(path, label):
    
    path = path.decode("utf-8")
    img = audio_to_img(path)
    label = np.int32(label)  
    return img, label

def tf_map(path, label):
    img, lab = tf.numpy_function(
        func=py_load,
        inp=[path, label],
        Tout=[tf.float32, tf.int32]   
    )
    img.set_shape((224,224,3))
    lab.set_shape(())
    return img, tf.one_hot(lab, NUM_CLASSES)

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(2000).map(tf_map, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_ds = val_ds.map(tf_map, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


xb, yb = next(iter(train_ds))
print("Batch X:", xb.shape, xb.dtype)
print("Batch Y:", yb.shape, yb.dtype)

Batch X: (32, 224, 224, 3) <dtype: 'float32'>
Batch Y: (32, 6) <dtype: 'float32'>


In [38]:
base = tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)
base.trainable = False

inputs = tf.keras.Input(shape=(224,224,3))
x = base(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │         7,686 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,265,670 (8.64 MB)

 Trainable params: 7,686 (30.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [39]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("FINAL_audio_event_model.keras", monitor="val_accuracy", save_best_only=True)
]

history = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks)

with open("class_names.json", "w", encoding="utf-8") as f:
    json.dump(CLASS_NAMES, f, indent=2)

print("✅ Saved FINAL_audio_event_model.keras")
print("✅ Saved class_names.json")

Epoch 1/15


100/100 ━━━━━━━━━━━━━━━━━━━━ 118s 1s/step - accuracy: 0.8935 - loss: 0.3302 - val_accuracy: 0.9748 - val_loss: 0.1113
Epoch 2/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 210s 2s/step - accuracy: 0.9748 - loss: 0.0989 - val_accuracy: 0.9824 - val_loss: 0.0753
Epoch 3/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 98s 976ms/step - accuracy: 0.9833 - loss: 0.0650 - val_accuracy: 0.9836 - val_loss: 0.0645
Epoch 4/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 97s 969ms/step - accuracy: 0.9871 - loss: 0.0495 - val_accuracy: 0.9849 - val_loss: 0.0539
Epoch 5/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 96s 959ms/step - accuracy: 0.9893 - loss: 0.0401 - val_accuracy: 0.9861 - val_loss: 0.0494
Epoch 6/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 203s 2s/step - accuracy: 0.9915 - loss: 0.0352 - val_accuracy: 0.9849 - val_loss: 0.0450
Epoch 7/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 97s 963ms/step - accuracy: 0.9924 - loss: 0.0308 - val_accuracy: 0.9861 - val_loss: 0.0417
Epoch 8/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 98s 981ms/step - accuracy: 0.9940 - loss: 0.0277 - val_accur

In [40]:

import json
import numpy as np
import librosa
import tensorflow as tf

# ---------- SAME FEATURE PIPELINE ----------
SAMPLE_RATE = 22050
DURATION = 4.0
N_MELS = 128
N_FFT = 2048
HOP = 512

def wav_to_model_input(wav_path: str):
    y, _ = librosa.load(wav_path, sr=SAMPLE_RATE, mono=True)

    # trim silence + normalize
    y, _ = librosa.effects.trim(y, top_db=25)
    y = librosa.util.normalize(y)

    # pad/trim
    target_len = int(SAMPLE_RATE * DURATION)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]

    # mel spectrogram (match training)
    S = librosa.feature.melspectrogram(
        y=y, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS
    )
    S_db = librosa.power_to_db(S, ref=np.max)

    # normalize 0..1
    S_db = (S_db - S_db.min()) / (S_db.max() - S_db.min() + 1e-9)

    # to image (224,224,3)
    img = S_db.astype(np.float32)[..., np.newaxis]
    img = tf.image.resize(img, (224, 224)).numpy()
    img = np.repeat(img, 3, axis=-1)

    # MobileNetV2 preprocess
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img * 255.0)

    return np.expand_dims(img, axis=0).astype(np.float32)

In [47]:
import numpy as np
import librosa
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report


SR = 22050
DURATION = 4
N_MELS = 128


def audio_to_img(path):
    y, _ = librosa.load(path, sr=SR, mono=True)

    target_len = int(SR * DURATION)  

    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]

    S = librosa.feature.melspectrogram(
        y=y, sr=SR, n_mels=N_MELS, n_fft=2048, hop_length=512
    )
    S_db = librosa.power_to_db(S, ref=np.max)

  
    S_db = (S_db - S_db.min()) / (S_db.max() - S_db.min() + 1e-9)

   
    img = S_db.astype(np.float32)[..., np.newaxis]
    img = tf.image.resize(img, (224, 224)).numpy()
    img = np.repeat(img, 3, axis=-1)

    
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img * 255.0)

    return img.astype(np.float32)


X_val_processed = []

for path in X_val:
    img = audio_to_img(path)
    X_val_processed.append(img)

X_val_processed = np.array(X_val_processed)

print("Processed shape:", X_val_processed.shape)


y_pred = model.predict(X_val_processed)


y_pred_classes = np.argmax(y_pred, axis=1)
y_true = y_val  

accuracy = accuracy_score(y_true, y_pred_classes)
print("\n Accuracy:", accuracy)

print("\n Classification Report:\n")
print(classification_report(y_true, y_pred_classes))

Processed shape: (794, 224, 224, 3)
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 517ms/step

 Accuracy: 0.9886649874055415

 Classification Report:

              precision    recall  f1-score   support

           0       0.99      1.00      1.00       144
           1       1.00      1.00      1.00       145
           3       1.00      0.50      0.67         2
           4       0.98      1.00      0.99       317
           5       0.99      0.96      0.98       186

    accuracy                           0.99       794
   macro avg       0.99      0.89      0.93       794
weighted avg       0.99      0.99      0.99       794

